# Exercise 06 — aggregating consumer & anomaly detection

**Goal:** build a consumer that doesn't just *print* events but **processes** them in real time — like the operations dashboard of the delivery app:

1. Parse the JSON payload.
2. Maintain running statistics per rider (count, min, max, average speed).
3. Flag GPS glitches (impossible speeds) as anomalies.
4. Print a summary table — and optionally update it live.

**Prerequisite:** run [`exercise_05_produce_stream.ipynb`](exercise_05_produce_stream.ipynb) (especially Task B — anomaly simulation) so there is data to aggregate.

## Background — state in a streaming context

In batch processing you load all data, then aggregate. In streaming you maintain *state* that updates as each event arrives. Our state lives in a few `defaultdict`s — that's enough for a single-process consumer.

Real production stream processors (Flink, Kafka Streams, …) replicate and persist this state so it survives crashes. We're keeping it in RAM for clarity.

In [ ]:
from confluent_kafka import Consumer
from collections import defaultdict
from datetime import datetime
import json

BROKER = 'redpanda:29092'

# Anything above this is treated as a GPS glitch:
# no bike or scooter in town does 80 km/h.
MAX_SPEED_KMH = 80.0

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'rider-stats',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe(['rider-locations'])
print('Consumer ready.')

## Step 1 — read and aggregate (up to 200 events)

Three accumulators:

- `totals[rider]` — sum of the speeds (used for the average).
- `counts[rider]` — number of pings (denominator).
- `anomalies` — list of pings whose speed exceeded the limit.

**Task:** complete the aggregation inside the loop.

In [ ]:
totals    = defaultdict(float)
counts    = defaultdict(int)
anomalies = []

messages_read, empty_polls = 0, 0
while messages_read < 200 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1

    try:
        data  = json.loads(msg.value().decode())
        rider = data.get('rider_id', 'unknown')
        speed = float(data.get('speed_kmh', 0.0))

        # TODO: update totals[rider] and counts[rider]


        # TODO: if speed > MAX_SPEED_KMH, append to anomalies:
        #   anomalies.append({'rider': rider, 'speed': speed,
        #                     'offset': msg.offset()})

    except Exception as e:
        print(f'Parse error: {e}')

print(f'Done — processed {messages_read} events.')

## Step 2 — print the summary table

Format:

```
Rider   | Count |      Avg
--------------------------
rider_3 |    12 | 17.34 km/h
```

**Task:** loop over `totals`, compute `avg = totals[r] / counts[r]`, and print one row per rider.

In [ ]:
# TODO: print a summary table

## Step 3 — anomaly report

List all detected anomalies. Show how far above the limit each is, in percent. Hint: `pct_over = (speed / MAX_SPEED_KMH - 1) * 100`.

If there are no anomalies, re-run the anomaly simulator in `exercise_05_produce_stream.ipynb` Task B and try again.

**Think about it:** the glitches also pull up the average speed in your table from Step 2. Should the dashboard drop flagged pings before averaging?

In [ ]:
# TODO: print each anomaly with rider, speed and percent over the limit

## Bonus — live aggregation

Build a continuous loop that prints an updated summary every 10 events. While it runs, fire the streaming producer in another tab and watch the averages converge.

Tip: `from IPython.display import clear_output; clear_output(wait=True)` lets you redraw the same area instead of scrolling forever.

In [ ]:
# TODO: live aggregation loop

In [ ]:
try:
    consumer.close()
except Exception:
    pass
print('Closed.')

## What you learned

- A *processing* consumer maintains in-memory state and updates it per message — fundamentally different from "just print".
- Threshold-based anomaly detection is dead simple but surprisingly effective when there is a hard physical limit (no bike does 80 km/h). Statistical methods (z-score, IQR) come later.
- This whole pattern — "per-key running aggregate" — is exactly what dedicated stream processors like Flink were built for. We'll meet Flink in the next track.

**Think about it:** a rider whose speed stays at 0 km/h for 15 minutes is probably stuck — a flat tyre, or a restaurant that is late. Why can't you detect that with a per-event threshold like above? What would you need to remember per rider, and for how long?